# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the [FAIR^2 dataset](https://doi.org/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following the Croissant FAIR data specification.

### Dataset Source
The dataset schema is available here:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

The dataset captures demographic, clinicopathological, and molecular characteristics of 77 cancer survivors with second primary colorectal cancer, supporting investigations of MSI-H status and anatomical distribution.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print metadata name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and all field and column `@id`s associated with the dataset.

In [ ]:
# List all record sets (by @id)
record_sets = dataset.metadata.recordSet
if not record_sets:
    print('No record sets found in metadata! The dataset might be flat/tabular.')
else:
    print('Record Sets:')
    for rset in record_sets:
        print(f"- {getattr(rset, '@id', None)}: {getattr(rset, 'name', None)}")

# For this dataset, if record sets are not directly declared, enumerate keys in dataset.records() generator
all_recordset_ids = []
# The mlcroissant API expects the record set's @id; let's try to extract one by listing record sets.
has_explicit_recordset = bool(record_sets)
if has_explicit_recordset:
    for rset in record_sets:
        rid = getattr(rset, '@id', None)
        if rid:
            all_recordset_ids.append(rid)
else:
    # Try to enumerate via generator (typically, 'records' returns an iterator over default record set)
    # Use mlc.Dataset._get_record_sets() as a last resort
    from mlcroissant._src.structure.dataset import _get_record_sets
    for rset in _get_record_sets(dataset.metadata):
        rid = getattr(rset, '@id', None)
        if rid:
            all_recordset_ids.append(rid)

if not all_recordset_ids:
    print("No record sets identified.\n")
else:
    print("\nRecord Sets detected:")
    for rid in all_recordset_ids:
        print(f"  - {rid}")

# For each record set, list its fields with their @id and name.
for rid in all_recordset_ids:
    print(f"\nFields for Record Set '{rid}':")
    rset_obj = None
    # Find the actual object with @id == rid
    if record_sets:
        for candidate in record_sets:
            if getattr(candidate, '@id', None) == rid:
                rset_obj = candidate
                break
    else:
        # Use _get_record_sets lookup
        for candidate in _get_record_sets(dataset.metadata):
            if getattr(candidate, '@id', None) == rid:
                rset_obj = candidate
                break
    
    if not rset_obj:
        print(f"  Could not locate record set object for {rid}")
        continue

    # List fields
    fields = getattr(rset_obj, 'field', [])
    if not fields:
        print("  (No fields defined)")
    else:
        for field in fields:
            print(f"  - {getattr(field, '@id', None)}: {getattr(field, 'name', None)}  ({getattr(field, 'dataType', None)})")

## 3. Data Extraction
Load each record set into a DataFrame for analysis. You'll use the record set and field `@id`s discovered above.

In [ ]:
# Prepare to extract data for each record set
dfs = {}

if not all_recordset_ids:
    # Default: try with 'default'/'main' record set
    print("No explicit record set IDs; attempting to load dataset as a flat table.")
    df = pd.DataFrame(dataset.records())
    dfs['main'] = df
    print(f"Loaded flat dataset with columns: {df.columns.tolist()}")
else:
    print("Extracting record sets as DataFrames...")
    for rid in all_recordset_ids:
        recs = list(dataset.records(record_set=rid))
        dfs[rid] = pd.DataFrame(recs)
        print(f"- Loaded '{rid}' ({len(dfs[rid])} rows, {len(dfs[rid].columns)} columns)")

# Display available columns from the main record set
main_recordset_id = all_recordset_ids[0] if all_recordset_ids else 'main'
print("\nColumns available in main record set:")
print(dfs[main_recordset_id].columns.tolist())
dfs[main_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter records, normalize numeric fields, and group by key attributes using `@id` for fields. We'll demonstrate this with a numeric variable (e.g., Age at second diagnosis) and anatomical location.

In [ ]:
# List the columns for reference
df = dfs[main_recordset_id]
print("Columns available:")
for col in df.columns:
    print(col)

# Example field @id names (you may adjust as needed by inspecting above):
# We'll suppose the field IDs are like 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fields/age_at_second_diagnosis'
# and 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fields/anatomical_location'.

# Let's find candidate numeric field and group/categorical field
numeric_candidates = [col for col in df.columns if 'age' in col.lower() and 'second' in col.lower()]
group_candidates = [col for col in df.columns if 'anatomic' in col.lower() or 'site' in col.lower()]

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    # fallback to any numeric column
    numeric_field_id = df.select_dtypes(include='number').columns[0]
print(f"Using numeric field: {numeric_field_id}")

if group_candidates:
    group_field_id = group_candidates[0]
else:
    # fallback: second available column
    non_numeric = df.select_dtypes(exclude='number').columns.tolist()
    group_field_id = non_numeric[0] if non_numeric else df.columns[0]
print(f"Using grouping field: {group_field_id}")

# Remove obviously invalid ages (e.g. <10, >110)
filtered_df = df[df[numeric_field_id].between(10, 110)]
print(f"Filtered records with {numeric_field_id} between 10 and 110 ({len(filtered_df)}/{len(df)} records):")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by anatomical location or grouping variable
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the age distribution and its relation to anatomical location.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Age distribution (histogram)
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field_id], bins=15, kde=True)
plt.xlabel('Age at Second CRC Diagnosis')
plt.title('Distribution of Age at Second CRC Diagnosis')
plt.show()

# Age by anatomical group (boxplot)
plt.figure(figsize=(7,5))
sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Age at Second CRC Diagnosis')
plt.title(f'Age at Second CRC Diagnosis by {group_field_id}')
plt.tight_layout()
plt.show()

## 6. Conclusion
We have demonstrated how to load, explore, and perform initial EDA on the FAIR^2 dataset using the `mlcroissant` library. After extracting record sets and using field `@id` references, we filtered, normalized, and visualized clinical variables such as age at second diagnosis and its anatomical distribution. This workflow enables further reproducible and transparent biomedical data analyses following FAIR principles.